# Predicting Product Sales
## 2. Data Checks

Before cleaning and modeling, we perform rigorous checks on the dataset's integrity, data type consistency, logical correctness, and temporal range.

Checks performed:
- Dataset shape and columns data types
- Missing (null) values in all fields
- Duplicate transactions
- Value range checks for continuous variables (age, quantity, price)
- Time-series sanity check (validating continuity of invoice dates)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RAW_DATA_PATH = r'../data/raw-data/customer_shopping_data.csv'
df = pd.read_csv(RAW_DATA_PATH)
print('Shape:', df.shape)
df.info()

### 2.1 Missing Values

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_df = pd.DataFrame({'Missing Count': null_counts, 'Missing %': null_pct})
print(null_df)

### 2.2 Duplicate Records

In [ ]:
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup} ({n_dup / len(df) * 100:.2f}%)')

### 2.3 Value Range and Logical Checks
We check whether numeric variables hold logical values:
- Age should be within a standard human range ($18 \le \text{age} \le 100$).
- Quantity should be strictly positive (quantity $> 0$).
- Price should be strictly positive (price $> 0$).

In [ ]:
invalid_age = df[(df['age'] < 18) | (df['age'] > 100)]
print(f'Invalid age count (< 18 or > 100): {len(invalid_age)}')

invalid_quantity = df[df['quantity'] <= 0]
print(f'Invalid quantity count (<= 0): {len(invalid_quantity)}')

invalid_price = df[df['price'] <= 0]
print(f'Invalid price count (<= 0): {len(invalid_price)}')

### 2.4 Temporal Range and Continuity Check
We verify if the invoices are continuous and look for any drops in data recording.

In [ ]:
df_temp = df.copy()
df_temp['invoice_date'] = pd.to_datetime(df_temp['invoice_date'], format='%d/%m/%Y')
print('Minimum invoice date:', df_temp['invoice_date'].min())
print('Maximum invoice date:', df_temp['invoice_date'].max())

df_temp['YearMonth'] = df_temp['invoice_date'].dt.to_period('M')
monthly_counts = df_temp.groupby('YearMonth').size()
print('\nMonthly invoice counts:')
print(monthly_counts)

### 2.5 Summary of Findings

| Check | Status | Finding |
|-------|--------|---------|
| Missing Values | Passed | No missing values across all columns. |
| Duplicate Rows | Passed | No duplicate rows found in the raw dataset. |
| Age range | Passed | All customers are aged between 18 and 69. |
| Quantity/Price range | Passed | No transactions with quantity <= 0 or price <= 0. |
| Incomplete Data | Flagged | March 2023 contains only 3 days of data, leading to a drastic drop in counts. |

Proceed to **Step 3: Data Cleaning**.